In [1]:
%load_ext autoreload
%autoreload 2

import scanpy as sc
import sys
import os
import anndata
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import anndata
import seaborn as sns
import torch
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

print("CUDA Available: ", torch.cuda.is_available())
print("Number of GPUs: ", torch.cuda.device_count())

CUDA Available:  True
Number of GPUs:  1


In [ ]:
data = sc.read_h5ad("../../../original_data/plant_all_days.h5ad")

In [3]:
print(data)
group_map = {'Nodule (21 dpi)': 21, 'Nodule (12 dpi)': 12, 'Root': 0}
data.obs['Group'] = data.obs['Sample_time'].map(group_map)
# print(data.obs[['Sample_time', 'Group']].drop_duplicates())

AnnData object with n_obs × n_vars = 26712 × 39337
    obs: 'n_genes', 'n_counts', 'Cluster', 'leiden_R', 'Cell type', 'Sample_time'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std', 'SNF', 'means_ForPickMock'
    uns: 'Cell type_colors'
    obsm: 'X_scvi', 'X_umap'
    layers: 'empty', 'normalize_log', 'raw'
    obsp: 'connectivities', 'distances'


In [4]:
# See unique time points
print(data.obs['Sample_time'].unique())

# See value counts
print(data.obs['Sample_time'].unique())

print(data.obs['Cell type'].unique())
print(data)
# make train data
plant_data_12 = data[data.obs['Sample_time'] == 'Nodule (12 dpi)'].copy()
plant_data_21 = data[data.obs['Sample_time'] == 'Nodule (21 dpi)'].copy()
plant_data_0 = data[data.obs['Sample_time'] == 'Root'].copy() 
print(plant_data_12.obs['Sample_time'].unique())
print(plant_data_21.obs['Sample_time'].unique())
print(plant_data_0.obs['Sample_time'].unique())

['Nodule (21 dpi)', 'Nodule (12 dpi)', 'Root']
Categories (3, object): ['Nodule (21 dpi)', 'Nodule (12 dpi)', 'Root']
['Nodule (21 dpi)', 'Nodule (12 dpi)', 'Root']
Categories (3, object): ['Nodule (21 dpi)', 'Nodule (12 dpi)', 'Root']
['2: Outer cortex', '12: Infected cells', '9: Vascular bundle', '4: Outer cortex', '0: Uninfected cells', ..., '6: Unknown', '10: Unknown', '3: Vascular bundle', '13: Unknown', '14: Unknown']
Length: 15
Categories (15, object): ['0: Uninfected cells', '1: Inner cortex', '2: Outer cortex', '3: Vascular bundle', ..., '11: Uninfected cells', '12: Infected cells', '13: Unknown', '14: Unknown']
AnnData object with n_obs × n_vars = 26712 × 39337
    obs: 'n_genes', 'n_counts', 'Cluster', 'leiden_R', 'Cell type', 'Sample_time', 'Group'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std', 'SNF', 'means_ForPickMock'
    uns: 'Cell type_colors'
    obsm: 'X_scvi', 'X_umap'

In [5]:

# print(plant_data_0)
# print(plant_data_12)
# print(plant_data_21)

In [6]:
plant_data_0.write_h5ad('plant_paper_day0_adata.h5ad')
plant_data_12.write_h5ad('plant_paper_day12_adata.h5ad')
plant_data_21.write_h5ad('plant_paper_day21_adata.h5ad')

In [ ]:
combined = anndata.concat([plant_data_12, plant_data_21])
print(combined.obs['Group'].unique())
combined.write_h5ad(f'plant_train_for_day0_adata.h5ad')
plant_data_0.write_h5ad('plant_test_day0_data.h5ad')